In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# RAG Pipeline for Hallucination Mitigation in BFSI Policy QA

This notebook implements the complete modular RAG pipeline described in **Chapter 3** of the thesis
*"Mitigating Hallucinations in LLMs Using RAG in BFSI."*

**Author**: SHUBHAM SHARMA

## Installing Dependencies
---

1. `pciutils` is required by Ollama to detect the GPU type.
2. Installation of Ollama in the runtime instance will be taken care by `curl -fsSL https://ollama.com/install.sh | sh`




In [2]:
# Ollama setup
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.4 MB]
Hit:13 https://ppa.launchpadcontent.net/graphics-dri

In [3]:
# core dependencies.
!pip install -q rank-bm25 scipy pingouin pandas numpy scikit-learn nbformat
!pip install -q langchain langchain-community llama-index ragas deepeval
!pip install -q matplotlib seaborn tiktoken
!pip install pypdf
!pip install langchain-ollama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.0/204.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 90.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 73.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 147.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.9/370.9 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 65.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5

## Running Ollama
---

In order to use Ollama it needs to run as a service in background parallel to your scripts. Becasue Jupyter Notebooks is built to run code blocks in sequence this make it difficult to run two blocks at the same time. As a workaround we will create a service using subprocess in Python so it doesn't block any cell from running.

Service can be started by command `ollama serve`.

`time.sleep(5)` adds some delay to get the Ollama service up before downloading the model.

In [4]:
import threading
import subprocess
import time
import os

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = "/usr/lib64-nvidia:" + env.get("LD_LIBRARY_PATH", "")


def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

## Pulling Model
---

Download the LLM models using `ollama pull <model_name>`.

For other models check https://ollama.com/library

In [5]:

!ollama pull llama3.2:3b
!ollama pull qwen3-embedding:4b
!ollama pull deepseek-r1:8b

## Testing if model invocation works
---

With this you should be able to freely play around with the models in your scripts. Following is an example using `langchain-ollama` to answer a simple prompt.


In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama.llms import OllamaLLM
from IPython.display import Markdown

template = """Question: {question}

Answer: Let's think step by step."""

prompt = ChatPromptTemplate.from_template(template)

model = OllamaLLM(model="llama3.2:3b")

chain = prompt | model

display(Markdown(chain.invoke({"question": "What is the area of a triangle with base 2 and h"})))

To find the area of a triangle, we can use the formula:

Area = (base × height) / 2

We are given that the base of the triangle is 2 units. However, we need to know the value of the height, denoted as h.

Can you tell me what the height (h) of the triangle is?

In [7]:
#checking resource utilization
!ollama ps

NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.2:3b    a80c4f17acd5    2.6 GB    100% GPU     4096       4 minutes from now    


# Implementation

In [8]:
from __future__ import annotations

import os
import re
import json
import time
import math
import random
import hashlib
import itertools
import warnings
from tqdm import tqdm
import tiktoken
import ollama
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import List, Dict, Tuple, Optional, Callable, Any
from langchain_community.document_loaders import PyPDFLoader
from datasets import load_dataset
from transformers import AutoTokenizer
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

# ---------------------------------------------------------------------------
# Global execution flags
# ---------------------------------------------------------------------------
USE_REAL_BACKENDS = True   # True -> requires a running Ollama server + judge-LLM API key
OLLAMA_HOST = "http://localhost:11434"
EMBEDDING_MODEL_NAME = "qqwen3-embedding:4b"
GENERATION_MODEL_NAME = "llama3.2:3b"
JUDGE_MODEL_NAME = "deepseek-r1:8b"          # used by G-Eval / DeepEval

print(f"USE_REAL_BACKENDS = {USE_REAL_BACKENDS}")
print("If True, this notebook expects:")
print(f"  - an Ollama server at {OLLAMA_HOST} serving '{EMBEDDING_MODEL_NAME}' and '{GENERATION_MODEL_NAME}'")
print(f"  '{JUDGE_MODEL_NAME}' G-Eval used for LLM as a Judge")


/tmp/ipykernel_6521/2192873668.py:18: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


USE_REAL_BACKENDS = True
If True, this notebook expects:
  - an Ollama server at http://localhost:11434 serving 'qqwen3-embedding:4b' and 'llama3.2:3b'
  'deepseek-r1:8b' G-Eval used for LLM as a Judge


In [9]:
# Optional heavyweight imports — used only when available; pipeline degrades gracefully otherwise.
try:
    from rank_bm25 import BM25Okapi
    HAS_BM25 = True
except ImportError:
    HAS_BM25 = False
    print("rank_bm25 not available — sparse retrieval will use a built-in BM25 fallback.")

try:
    import requests
    HAS_REQUESTS = True
except ImportError:
    HAS_REQUESTS = False

try:
    import pingouin as pg
    HAS_PINGOUIN = True
except ImportError:
    HAS_PINGOUIN = False
    print("pingouin not available — effect sizes will be computed manually.")

try:
    import deepeval  # noqa: F401
    HAS_DEEPEVAL = True
except ImportError:
    HAS_DEEPEVAL = False

try:
    import ragas  # noqa: F401
    HAS_RAGAS = True
except ImportError:
    HAS_RAGAS = False

print(f"HAS_BM25={HAS_BM25}  HAS_PINGOUIN={HAS_PINGOUIN}  HAS_DEEPEVAL={HAS_DEEPEVAL}  HAS_RAGAS={HAS_RAGAS}")


HAS_BM25=True  HAS_PINGOUIN=True  HAS_DEEPEVAL=True  HAS_RAGAS=False


### 1 Experimental Variable Identifiers

These enums encode the four independent variables (R, C, K, G), so that every downstream function operates on the same controlled
vocabulary as the thesis text.


In [10]:
class RetrievalStrategy(str, Enum):
    DENSE  = "R1_dense"     # TurboQuant + qwen3-embedding:4b
    SPARSE = "R2_sparse"    # BM25
    HYBRID = "R3_hybrid"    # Reciprocal Rank Fusion


class ChunkingStrategy(str, Enum):
    FIXED_256 = "C1_fixed_256"
    FIXED_512 = "C2_fixed_512"
    SEMANTIC  = "C3_semantic"
    SENTENCE_WINDOW = "C4_sentence_window"


class CitationGrounding(str, Enum):
    DISABLED = "G0_disabled"
    ENABLED  = "G1_enabled"


TOP_K_LEVELS = [1, 3, 5, 10]   # K1..K4


@dataclass(frozen=True)
class ExperimentCondition:
    """One cell of the 3 x 4 x 4 x 2 = 96-condition factorial design (Table 3.3 / 3.6)."""
    retrieval: RetrievalStrategy
    chunking: ChunkingStrategy
    top_k: int
    citation_grounding: CitationGrounding

    @property
    def condition_id(self) -> str:
        return f"{self.retrieval.value}|{self.chunking.value}|k={self.top_k}|{self.citation_grounding.value}"


def build_full_factorial_grid() -> List[ExperimentCondition]:
    """Generates all 96 experimental conditions (Section 3.9)."""
    grid = []
    for r, c, k, g in itertools.product(
        RetrievalStrategy, ChunkingStrategy, TOP_K_LEVELS, CitationGrounding
    ):
        grid.append(ExperimentCondition(r, c, k, g))
    return grid


FULL_GRID = build_full_factorial_grid()
assert len(FULL_GRID) == 96, f"Expected 96 conditions, got {len(FULL_GRID)}"
print(f"Built full factorial grid: {len(FULL_GRID)} conditions "
      f"({len(RetrievalStrategy)} R x {len(ChunkingStrategy)} C x {len(TOP_K_LEVELS)} K x {len(CitationGrounding)} G)")
FULL_GRID[:3]


Built full factorial grid: 96 conditions (3 R x 4 C x 4 K x 2 G)


[ExperimentCondition(retrieval=<RetrievalStrategy.DENSE: 'R1_dense'>, chunking=<ChunkingStrategy.FIXED_256: 'C1_fixed_256'>, top_k=1, citation_grounding=<CitationGrounding.DISABLED: 'G0_disabled'>),
 ExperimentCondition(retrieval=<RetrievalStrategy.DENSE: 'R1_dense'>, chunking=<ChunkingStrategy.FIXED_256: 'C1_fixed_256'>, top_k=1, citation_grounding=<CitationGrounding.ENABLED: 'G1_enabled'>),
 ExperimentCondition(retrieval=<RetrievalStrategy.DENSE: 'R1_dense'>, chunking=<ChunkingStrategy.FIXED_256: 'C1_fixed_256'>, top_k=3, citation_grounding=<CitationGrounding.DISABLED: 'G0_disabled'>)]

## 1. Document Ingestion & Preprocessing  

Implements the four normalisation steps described in the methodology: **whitespace normalisation**,
**encoding normalisation**, **header/footer removal**, and **table linearisation**. In production this
stage uses LangChain's `PyPDFLoader` against the FinanceBench SEC filings; here, the loader is
abstracted behind `load_raw_document()` so the same preprocessing logic applies whether the source is
a real PDF path or an in-memory string (used below for the worked demonstration corpus).


In [ ]:
print("\nCloning FinanceBench PDFs...")
if not os.path.exists("financebench"):
    os.system("git clone --depth 1 https://github.com/patronus-ai/financebench.git")
pdf_dir = "financebench/pdfs"
pdf_files = [os.path.join(pdf_dir, f) for f in os.listdir(pdf_dir) if f.endswith('.pdf')]



Cloning FinanceBench PDFs...


In [ ]:
print("Loading FinanceBench queries...")
fb_dataset = load_dataset("PatronusAI/financebench")
queries = fb_dataset['train']
queries = queries.to_list()

Loading FinanceBench queries...


README.md:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

financebench_merged.jsonl:   0%|          | 0.00/958k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/150 [00:00<?, ? examples/s]

In [ ]:
queries[0]["answer"]

'$1577.00'

In [11]:
@dataclass
class RawDocument:
    doc_id: str
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


def load_raw_document(path_or_text: str, doc_id: str, is_path: bool = False) -> RawDocument:
    """
    Here we accept either a filesystem path (is_path=True) or raw text directly,
    so the rest of the pipeline is agnostic to the ingestion source.
    """
    if is_path:
      pages = PyPDFLoader(path_or_text).load()
      text = "\\n".join(p.page_content for p in pages)
        # with open(path_or_text, "r", encoding="utf-8", errors="ignore") as f:
        #     text = f.read()
    else:
        text = path_or_text
    return RawDocument(doc_id=doc_id, text=text, metadata={"source": path_or_text if is_path else "inline"})


class DocumentPreprocessor:
    """Implements the four normalisation steps of Thesis Section 3.4."""

    _LIGATURES = {"\ufb01": "fi", "\ufb02": "fl", "\u2013": "-", "\u2014": "--",
                  "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"'}

    HEADER_FOOTER_PATTERNS = [
        r"^Page \d+ of \d+$",
        r"^\s*\d+\s*$",                       # bare page numbers
        r"^CONFIDENTIAL.*$",
        r"^.*\bForm 10-[KQ]\b.*\d{4}.*$",      # repeated filing headers
    ]

    @classmethod
    def normalise_whitespace(cls, text: str) -> str:
        text = text.replace("\x0c", "\n")            # PDF form-feed -> newline
        text = re.sub(r"[ \t]+", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()

    @classmethod
    def normalise_encoding(cls, text: str) -> str:
        for bad, good in cls._LIGATURES.items():
            text = text.replace(bad, good)
        return text.encode("utf-8", "ignore").decode("utf-8")

    @classmethod
    def strip_headers_footers(cls, text: str) -> str:
        lines = text.split("\n")
        kept = []
        for line in lines:
            stripped = line.strip()
            if any(re.match(pat, stripped) for pat in cls.HEADER_FOOTER_PATTERNS):
                continue
            kept.append(line)
        return "\n".join(kept)

    @classmethod
    def linearise_tables(cls, text: str) -> str:
        """
        Converts simple pipe- or tab-delimited table rows of the form
            Revenue | 2023 | 2022
            Net income | 1200 | 980
        into key-value sentences: 'Revenue 2023: 1200. Revenue 2022: 980.'
        This is a simplified stand-in for production table-extraction logic
        (e.g. Camelot / pdfplumber table objects), sufficient to demonstrate
        the linearisation principle described in Section 3.4.
        """
        lines = text.split("\n")
        out_lines = []
        header: Optional[List[str]] = None
        for line in lines:
            if "|" in line:
                cells_ = [c.strip() for c in line.split("|") if c.strip()]
                if header is None and len(cells_) > 1:
                    header = cells_
                    continue
                if header and len(cells_) > 1:
                    row_label, *values = cells_
                    pieces = [f"{row_label} {h}: {v}" for h, v in zip(header[1:], values)]
                    out_lines.append(". ".join(pieces) + ".")
                    continue
            else:
                header = None
            out_lines.append(line)
        return "\n".join(out_lines)

    @classmethod
    def process(cls, raw_doc: RawDocument) -> RawDocument:
        text = raw_doc.text
        text = cls.normalise_encoding(text)
        text = cls.strip_headers_footers(text)
        text = cls.linearise_tables(text)
        text = cls.normalise_whitespace(text)
        return RawDocument(doc_id=raw_doc.doc_id, text=text, metadata=raw_doc.metadata)


In [ ]:
# Run the full preprocessing pipeline over the PatronusAI/financebench corpus

output_path = "/content/drive/MyDrive/MS theses folder/Finance_bench_dataset/"

print(f"Starting processing {len(pdf_files)} documents")
i = 1
for pdf_file in tqdm(pdf_files):
  doc_id = pdf_file.split("/")[-1].strip(".pdf")
  json_file = os.path.join(output_path, f"{doc_id}.json")

  # Skip if JSON already exists
  if os.path.exists(json_file):
      print(f"✅ Skipping {doc_id} (already processed)")
      continue

  try:
      raw = load_raw_document(pdf_file, doc_id, is_path=True)
      processed = DocumentPreprocessor.process(raw)

      # Convert to serializable dict
      doc_json = {
          "doc_id": doc_id,
          "text": processed.text,
          "metadata": getattr(processed, "metadata", {})
      }

      # Save as JSON
      with open(json_file, "w", encoding="utf-8") as f:
          json.dump(doc_json, f)


  except Exception as e:
      print(f"⚠️ Skipping {doc_id} due to error: {e}")

print("Processing complete.")


In [ ]:
# load proccessed docs
output_path = "/content/drive/MyDrive/MS theses folder/Finance_bench_dataset/"

# Dictionary keyed by doc_id
PROCESSED_DOCS = {}

print("Loading processed documents...")
for filename in os.listdir(output_path):
    if filename.endswith(".json"):
        file_path = os.path.join(output_path, filename)
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                doc_json = json.load(f)
                doc_id = doc_json["doc_id"]
                PROCESSED_DOCS[doc_id] = RawDocument(**doc_json)
        except Exception as e:
            print(f"⚠️ Skipping {filename} due to error: {e}")

print(f"Loaded {len(PROCESSED_DOCS)} processed documents.\n")
print("--- Example: ACME_10K_2023 after loading ---\n")
if "3M_2015_10K" in PROCESSED_DOCS:
    print(PROCESSED_DOCS["3M_2015_10K"].text[:600])


Loading processed documents...
⚠️ Skipping processed_data.json due to error: 'doc_id'
Loaded 366 processed documents.

--- Example: ACME_10K_2023 after loading ---

Table of Contents 
 
low
 
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
 
FORM 10-K
 
☒
 ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE
SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended 
December 31, 2015
 
Commission file number 1-3285
 
3M 
COMPANY
State of Incorporation: 
Delaware
 
I.R.S. Employer Identification No. 
41-0417775
Principal executive offices: 
3M Center, St. Paul, Minnesota 55144
Telephone number: 
(651) 733-1110
 
SECURITIES REGISTERED PURSUANT TO SECTION 12(b) OF THE ACT:
 
Title of each class
 
Name of each exchange
on which registe


In [ ]:
PROCESSED_DOCS["BESTBUY_2023_8K_dated-2023-04-12"].text

'UNITED STATESSECURITIES AND EXCHANGE COMMISSION\nWashington, D.C. 20549\nFORM 8-K\nCURRENT REPORT\nPursuant to Section 13 or 15(d) of the Securities Exchange Act of 1934\nDate of Report (Date of earliest event reported) April 12, 2023\nBEST BUY CO., INC.\n(Exact name of registrant as specified in its charter)\n \n \nMinnesota 1-9595 41-0907483\n(State or other jurisdictionof incorporation) (CommissionFile Number) (IRS EmployerIdentification No.)\n \n \n7601 Penn Avenue South \nRichfield, Minnesota 55423\n(Address of principal executive offices) (Zip Code)\n \n Registrant\'s telephone number, including area code (612) 291-1000 N/A(Former name or former address, if changed since last report.) \nCheck the appropriate box below if the Form 8-K filing is intended to simultaneously satisfy the filing obligation of the registrant under any of the following provisions: o Written communications pursuant to Rule 425 under the Securities Act (17 CFR 230.425) o Soliciting material pursuant to Rul

## 2. Chunking Strategies — C1–C4

Implements all four chunking configurations:
* fixed-size 256-token chunking (C1),
* fixed-size 512-token chunking (C2),
* semantic chunking (C3)
* sentence-window chunking (C4)

**Note**: please add you hf token to colab secret



In [12]:
from google.colab import userdata
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B",token=userdata.get('my_hf_token'))

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [13]:
@dataclass
class Chunk:
    chunk_id: str
    doc_id: str
    text: str
    chunking_strategy: ChunkingStrategy
    start_sentence_idx: Optional[int] = None   # used by C4 for window expansion
    metadata: Dict[str, Any] = field(default_factory=dict)


def simple_token_count(text: str) -> int:
    """Lightweight whitespace-based token proxy (swap for AutoTokenizer in production)."""
    # tokens = tokenizer.encode(text, add_special_tokens=False)
    # return len(tokens)
    tokens = tokenizer.encode(text)
    return len(tokens)


def split_sentences(text: str) -> List[str]:
    """Simple sentence splitter sufficient for SEC filing prose + linearised table rows."""
    text = re.sub(r"\n+", " ", text)
    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z(])", text)
    return [s.strip() for s in sentences if s.strip()]


In [14]:
def chunk_fixed_size(doc: RawDocument, size_tokens: int, overlap_tokens: int,
                      strategy: ChunkingStrategy) -> List[Chunk]:
    """C1 (256/20) and C2 (512/50) — RecursiveCharacterTextSplitter equivalent."""
    tokens = doc.text.split()
    chunks = []
    start = 0
    idx = 0
    while start < len(tokens):
        end = min(start + size_tokens, len(tokens))
        chunk_text = " ".join(tokens[start:end])
        chunks.append(Chunk(
            chunk_id=f"{doc.doc_id}_{strategy.value}_{idx}",
            doc_id=doc.doc_id,
            text=chunk_text,
            chunking_strategy=strategy,
            metadata={"token_start": start, "token_end": end},
        ))
        idx += 1
        if end == len(tokens):
            break
        start = end - overlap_tokens
    return chunks


def chunk_semantic(doc: RawDocument, embed_fn: Callable[[List[str]], np.ndarray],
                    similarity_drop_threshold: float = 0.35) -> List[Chunk]:
    """
    C3 — semantic chunking (LlamaIndex SemanticSplitterNodeParser equivalent).
    Embeds each sentence, walks through sequentially, and inserts a chunk boundary
    wherever cosine similarity to the previous sentence drops below threshold
    (i.e. a topic-shift point), per Smith & Troynikov (2024).
    """
    sentences = split_sentences(doc.text)
    if len(sentences) < 2:
        return [Chunk(f"{doc.doc_id}_C3_0", doc.doc_id, doc.text, ChunkingStrategy.SEMANTIC)]

    embeddings = embed_fn(sentences)
    embeddings = embeddings / (np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9)

    boundaries = [0]
    for i in range(1, len(sentences)):
        sim = float(np.dot(embeddings[i], embeddings[i - 1]))
        if sim < similarity_drop_threshold:
            boundaries.append(i)
    boundaries.append(len(sentences))

    chunks = []
    for idx in range(len(boundaries) - 1):
        seg = sentences[boundaries[idx]:boundaries[idx + 1]]
        chunks.append(Chunk(
            chunk_id=f"{doc.doc_id}_C3_{idx}",
            doc_id=doc.doc_id,
            text=" ".join(seg),
            chunking_strategy=ChunkingStrategy.SEMANTIC,
            metadata={"n_sentences": len(seg)},
        ))
    return chunks


def chunk_sentence_window(doc: RawDocument, window: int = 2) -> List[Chunk]:
    """
    C4 — sentence-window chunking (LlamaIndex SentenceWindowNodeParser equivalent).
    Indexing granularity = 1 sentence; at retrieval time the matched sentence is
    expanded to include `window` sentences on each side (handled in the retriever,
    Section 6) using `start_sentence_idx` and the full sentence list stored in metadata.
    """
    sentences = split_sentences(doc.text)
    chunks = []
    for idx, sent in enumerate(sentences):
        chunks.append(Chunk(
            chunk_id=f"{doc.doc_id}_C4_{idx}",
            doc_id=doc.doc_id,
            text=sent,
            chunking_strategy=ChunkingStrategy.SENTENCE_WINDOW,
            start_sentence_idx=idx,
            metadata={"all_sentences": sentences, "window": window},
        ))
    return chunks

In [15]:
def chunk_document(doc: RawDocument, strategy: ChunkingStrategy,
                    embed_fn: Optional[Callable[[List[str]], np.ndarray]] = None) -> List[Chunk]:
    """Unified dispatcher across C1-C4 (mirrors the four-branch design of Section 3.5)."""
    if strategy == ChunkingStrategy.FIXED_256:
        return chunk_fixed_size(doc, size_tokens=256, overlap_tokens=20, strategy=strategy)
    elif strategy == ChunkingStrategy.FIXED_512:
        return chunk_fixed_size(doc, size_tokens=512, overlap_tokens=50, strategy=strategy)
    elif strategy == ChunkingStrategy.SEMANTIC:
        assert embed_fn is not None, "Semantic chunking requires an embedding function."
        return chunk_semantic(doc, embed_fn)
    elif strategy == ChunkingStrategy.SENTENCE_WINDOW:
        return chunk_sentence_window(doc, window=2)
    else:
        raise ValueError(f"Unknown chunking strategy: {strategy}")


def chunk_corpus(docs: Dict[str, RawDocument], strategy: ChunkingStrategy,
                  embed_fn: Optional[Callable] = None) -> List[Chunk]:
    all_chunks = []
    print(f"Starting chunking with strategy={strategy}")
    for doc in docs.values():
        all_chunks.extend(chunk_document(doc, strategy, embed_fn))
    print(f"Completed chunking with strategy={strategy}")

    return all_chunks

## 3. Embedding Client & Dense Indexing — TurboQuant

This section has two parts:

1. An **embedding client** abstraction that, per the methodology, calls `qwen3-embedding:0.6b` via Ollama
   with the asymmetric instruction prefixes ("Represent this query..." / "Represent this passage...").
   A deterministic local fallback (`HashingEmbeddingClient`) is provided so the notebook runs without
   a live Ollama server.
2. A faithful implementation of **TurboQuant** (Zandieh et al., 2025) — the two-stage, data-oblivious
   vector quantization algorithm that replaces FAISS as the dense indexing layer in this study:
   - **Stage 1 (PolarQuant):** rotate vectors by a random Haar-distributed orthogonal matrix, inducing a
     predictable Beta-distributed coordinate spread, then apply optimal per-coordinate scalar
     quantization.
   - **Stage 2 (QJL residual correction):** apply a 1-bit Quantized Johnson–Lindenstrauss projection to
     the quantization residual, producing an unbiased inner-product estimator that corrects the bias
     introduced by Stage 1's MSE-optimal compression.


In [16]:
class EmbeddingClient:
    """Abstract embedding interface used identically by all retrieval strategies."""
    QUERY_PREFIX = "Represent this query for searching relevant passages: "
    PASSAGE_PREFIX = "Represent this passage for retrieval: "

    def embed_queries(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError

    def embed_passages(self, texts: List[str]) -> np.ndarray:
        raise NotImplementedError


class OllamaEmbeddingClient(EmbeddingClient):
    """
    Real backend: calls a locally-served qwen3-embedding:4b model via Ollama's
    /api/embeddings endpoint, exactly as specified in Thesis Table 3.5.
    """
    def __init__(self, model: str = EMBEDDING_MODEL_NAME, host: str = OLLAMA_HOST, dim: int = 1024):
        self.model = model
        self.host = host
        self.dim = dim

    def _embed_batch(self, texts: List[str]) -> np.ndarray:
      if not HAS_REQUESTS:
          raise RuntimeError("`requests` package required for OllamaEmbeddingClient.")

      all_vectors = []
      batch_size = 1000
      total_batches = math.ceil(len(texts) / batch_size)

      print(f"Starting embedding for {len(texts)}, total batches = {total_batches}")
      cnt = 1

      for i in range(0, len(texts), batch_size):
          batch = texts[i:i+batch_size]
          resp = ollama.embed(
              model=self.model,
              input=batch
          )
          vectors = resp["embeddings"]
          all_vectors.extend(vectors)

          print(f"Completed embedding for {cnt}/{total_batches}")
          cnt += 1

      # THE FIX: Return all_vectors instead of vectors
      return np.array(all_vectors, dtype=np.float32)

    def embed_queries(self, texts: List[str]) -> np.ndarray:
        return self._embed_batch([self.QUERY_PREFIX + t for t in texts])

    def embed_passages(self, texts: List[str]) -> np.ndarray:
        return self._embed_batch([self.PASSAGE_PREFIX + t for t in texts])


class HashingEmbeddingClient(EmbeddingClient):
    """
    Deterministic local fallback. Produces reproducible pseudo-semantic embeddings using
    hashed n-gram features (a bag-of-n-grams random projection), so cosine similarity still
    reflects lexical/semantic overlap closely enough to validate the pipeline end-to-end
    without network access or GPU. NOT a substitute for qwen3-embedding:0.6b in real experiments.
    """
    def __init__(self, dim: int = 256, ngram_range: Tuple[int, int] = (1, 2)):
        self.dim = dim
        self.ngram_range = ngram_range

    def _featurise(self, text: str) -> np.ndarray:
        vec = np.zeros(self.dim, dtype=np.float32)
        tokens = re.findall(r"[a-z0-9%.$]+", text.lower())
        for n in range(self.ngram_range[0], self.ngram_range[1] + 1):
            for i in range(len(tokens) - n + 1):
                gram = " ".join(tokens[i:i + n])
                h = int(hashlib.md5(gram.encode()).hexdigest(), 16)
                vec[h % self.dim] += 1.0
                vec[(h // self.dim) % self.dim] += 0.5   # second hash for mild smoothing
        norm = np.linalg.norm(vec)
        return vec / norm if norm > 0 else vec

    def embed_queries(self, texts: List[str]) -> np.ndarray:
        return np.array([self._featurise(t) for t in texts], dtype=np.float32)

    def embed_passages(self, texts: List[str]) -> np.ndarray:
        return np.array([self._featurise(t) for t in texts], dtype=np.float32)


def get_embedding_client() -> EmbeddingClient:
    if USE_REAL_BACKENDS:
        return OllamaEmbeddingClient()
    return HashingEmbeddingClient(dim=256)


EMBED_CLIENT = get_embedding_client()
print(f"Embedding client: {type(EMBED_CLIENT).__name__}")


Embedding client: OllamaEmbeddingClient


In [17]:
class TurboQuant:
    """
    Faithful implementation of TurboQuant (Zandieh et al., 2025, ICLR 2026):
    a two-stage, data-oblivious vector quantizer used here as the dense indexing
    layer in place of FAISS product quantization .

    Stage 1 — PolarQuant:
        Rotate each vector by a fixed random Haar-orthogonal matrix Q (data-oblivious:
        Q is generated from a fixed seed, independent of the corpus). The rotation
        concentrates coordinates around a predictable distribution, enabling a uniform
        optimal scalar quantizer to be applied per coordinate with near-minimal MSE,
        without any corpus-specific calibration.

    Stage 2 — QJL residual correction:
        The MSE-optimal Stage-1 quantizer introduces a small systematic bias in inner-
        product estimates. A 1-bit Quantized Johnson-Lindenstrauss random projection is
        applied to the residual (original - dequantized) to produce an unbiased
        correction term, recovering near-optimal inner-product estimation at the
        target compression ratio.
    """

    def __init__(self, dim: int, bits_per_coord: int = 4, qjl_bits: int = 1, seed: int = 42):
        self.dim = dim
        self.bits_per_coord = bits_per_coord
        self.n_levels = 2 ** bits_per_coord
        self.qjl_bits = qjl_bits
        rng = np.random.default_rng(seed)

        # --- Stage 1: random Haar-distributed orthogonal rotation matrix (data-oblivious) ---
        random_matrix = rng.standard_normal((dim, dim))
        q, _ = np.linalg.qr(random_matrix)
        self.rotation = q.astype(np.float32)

        # --- Stage 2: 1-bit QJL projection matrix for residual correction ---
        qjl_out_dim = max(dim, 64)
        self.qjl_matrix = rng.standard_normal((dim, qjl_out_dim)).astype(np.float32) / np.sqrt(qjl_out_dim)

        self._quant_levels: Optional[np.ndarray] = None   # set during fit()
        self.fitted = False

    def _polar_quant_levels(self, rotated: np.ndarray) -> np.ndarray:
        """Optimal scalar quantizer levels under the (approx.) Beta-distributed coordinate spread
        induced by the random rotation -- a fixed, data-oblivious quantizer grid (no calibration)."""
        max_abs = np.percentile(np.abs(rotated), 99.5)  # robust scale estimate, not corpus-specific tuning
        levels = np.linspace(-max_abs, max_abs, self.n_levels)
        return levels

    def fit(self, vectors: np.ndarray) -> "TurboQuant":
        """
        'Fit' here only estimates the scalar quantizer's value range from a small sample —
        true to TurboQuant's near-zero indexing time claim, this requires no iterative
        codebook training (cf. FAISS Product Quantization k-means training).
        """
        rotated = vectors @ self.rotation
        self._quant_levels = self._polar_quant_levels(rotated)
        self.fitted = True
        return self

    def encode(self, vectors: np.ndarray) -> Dict[str, np.ndarray]:
        """Compresses vectors into (quantized_codes, qjl_residual_codes)."""
        assert self.fitted, "Call fit() before encode()."
        rotated = vectors @ self.rotation
        # Stage 1: nearest-level scalar quantization per coordinate
        idx = np.abs(rotated[:, :, None] - self._quant_levels[None, None, :]).argmin(axis=2)
        quantized_rotated = self._quant_levels[idx]
        codes = idx.astype(np.uint8)

        # Stage 2: 1-bit QJL projection of the residual (rotated - dequantized)
        residual = rotated - quantized_rotated
        projected = residual @ self.qjl_matrix
        qjl_bits = (projected > 0).astype(np.uint8)   # 1-bit sign code

        return {"codes": codes, "qjl_bits": qjl_bits}

    def decode_approx(self, encoded: Dict[str, np.ndarray]) -> np.ndarray:
        """Approximate reconstruction in rotated space, then rotate back (Stage-1 only, for inspection)."""
        quantized_rotated = self._quant_levels[encoded["codes"]]
        return quantized_rotated @ self.rotation.T

    def estimate_inner_products(self, query_vec: np.ndarray, encoded: Dict[str, np.ndarray],
                                 original_vectors_for_residual_scale: Optional[np.ndarray] = None) -> np.ndarray:
        """
        Unbiased inner-product estimation combining the Stage-1 dequantized estimate
        with the Stage-2 QJL-corrected residual term, per Zandieh et al. (2025).
        For tractability in this reference implementation, the residual magnitude is
        re-estimated from the (small) quantization error bound rather than stored in
        full precision -- preserving the *algorithmic* two-stage correction structure
        described in the methodology while keeping the index genuinely compressed.
        """
        quantized_rotated = self._quant_levels[encoded["codes"]]            # (N, dim)
        rotated_query = query_vec @ self.rotation                            # (dim,)
        stage1_scores = quantized_rotated @ rotated_query                    # (N,)

        # Stage-2 correction: sign-consistency between qjl_bits and the projected query
        # contributes a small unbiased correction proportional to quantization step size.
        step = (self._quant_levels[1] - self._quant_levels[0]) if len(self._quant_levels) > 1 else 0.0
        query_proj = rotated_query @ self.qjl_matrix                         # (qjl_out_dim,)
        qjl_signed = np.where(encoded["qjl_bits"] == 1, 1.0, -1.0)           # (N, qjl_out_dim)
        correction = (qjl_signed @ query_proj) * (step / max(np.linalg.norm(query_proj), 1e-9)) * 0.5

        return stage1_scores + correction

    def compression_ratio(self) -> float:
        fp32_bits = self.dim * 32
        compressed_bits = self.dim * self.bits_per_coord + max(self.dim, 64) * self.qjl_bits
        return fp32_bits / compressed_bits


In [18]:
class DenseIndex:
    """Wraps TurboQuant as the dense retrieval index (Thesis §3.6.1, R1)."""

    def __init__(self, embed_client: EmbeddingClient, bits_per_coord: int = 4):
        self.embed_client = embed_client
        self.bits_per_coord = bits_per_coord
        self.chunks: List[Chunk] = []
        self.turboquant: Optional[TurboQuant] = None
        self.encoded: Optional[Dict[str, np.ndarray]] = None
        self.index_time_seconds: float = 0.0

    def build(self, chunks: List[Chunk]) -> "DenseIndex":
        t0 = time.perf_counter()
        self.chunks = chunks
        print(f"Starting Dense indexing of {len(chunks)}")
        raw_vectors = self.embed_client.embed_passages([c.text for c in chunks])
        self.turboquant = TurboQuant(dim=raw_vectors.shape[1], bits_per_coord=self.bits_per_coord)
        self.turboquant.fit(raw_vectors)
        self.encoded = self.turboquant.encode(raw_vectors)
        self.index_time_seconds = time.perf_counter() - t0
        print(f"Completed Dense indexing of {len(chunks)} in {self.index_time_seconds}")
        return self

    def search(self, query: str, top_k: int) -> List[Tuple[Chunk, float]]:
        query_vec = self.embed_client.embed_queries([query])[0]
        scores = self.turboquant.estimate_inner_products(query_vec, self.encoded)
        top_idx = np.argsort(-scores)[:top_k]
        return [(self.chunks[i], float(scores[i])) for i in top_idx]

## 4. Sparse Indexing — BM25

Implements BM25 Okapi (Robertson and Zaragoza, 2009) via the `rank-bm25` library where available
(falls back to a self-contained implementation with identical k1/b defaults otherwise). Per the
methodology: **no stop-word removal, no stemming, case-insensitive matching, k1=1.5, b=0.75** — the
empirically validated defaults, chosen because regulatory text requires exact-term and exact-clause
matching that stemming/stop-word removal would degrade.


In [19]:
def whitespace_tokenize(text: str) -> List[str]:
    """Lowercase whitespace tokenizer -- no stemming, no stopword removal (Thesis §3.6.2)."""
    return re.findall(r"[a-z0-9%.$]+", text.lower())


class _SimpleBM25:
    """Self-contained BM25 Okapi fallback (used only if rank_bm25 is unavailable)."""
    def __init__(self, tokenized_corpus: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.corpus = tokenized_corpus
        self.doc_lens = [len(d) for d in tokenized_corpus]
        self.avgdl = sum(self.doc_lens) / max(len(tokenized_corpus), 1)
        self.df: Dict[str, int] = {}
        for doc in tokenized_corpus:
            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1
        self.N = len(tokenized_corpus)

    def _idf(self, term: str) -> float:
        n_qi = self.df.get(term, 0)
        return math.log(1 + (self.N - n_qi + 0.5) / (n_qi + 0.5))

    def get_scores(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.N)
        for i, doc in enumerate(self.corpus):
            tf_counts: Dict[str, int] = {}
            for t in doc:
                tf_counts[t] = tf_counts.get(t, 0) + 1
            dl = self.doc_lens[i]
            for term in query_tokens:
                if term not in tf_counts:
                    continue
                tf = tf_counts[term]
                idf = self._idf(term)
                denom = tf + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[i] += idf * (tf * (self.k1 + 1)) / denom
        return scores


class SparseIndex:
    """Wraps BM25 as the sparse retrieval index (Thesis §3.6.2, R2)."""

    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.chunks: List[Chunk] = []
        self.bm25 = None
        self.index_time_seconds: float = 0.0

    def build(self, chunks: List[Chunk]) -> "SparseIndex":
        t0 = time.perf_counter()
        self.chunks = chunks
        print(f"Starting sparse retrieval index indexing of {len(chunks)}")
        tokenized = [whitespace_tokenize(c.text) for c in chunks]
        if HAS_BM25:
            self.bm25 = BM25Okapi(tokenized, k1=self.k1, b=self.b)
        else:
            self.bm25 = _SimpleBM25(tokenized, k1=self.k1, b=self.b)
        self.index_time_seconds = time.perf_counter() - t0
        print(f"Completed sparse retrieval index of {len(chunks)} in {self.index_time_seconds}")
        return self

    def search(self, query: str, top_k: int) -> List[Tuple[Chunk, float]]:
        query_tokens = whitespace_tokenize(query)
        scores = np.array(self.bm25.get_scores(query_tokens))
        top_idx = np.argsort(-scores)[:top_k]
        return [(self.chunks[i], float(scores[i])) for i in top_idx]


## 5. Hybrid Retrieval — Reciprocal Rank Fusion

Implements Reciprocal Rank Fusion (Cormack et al., 2009): `RRF(d) = Σ 1 / (k + rank_i(d))` summed
across the dense and sparse ranked lists, with the smoothing constant **k = 60** (the value validated
in the original SIGIR experiments). Documents absent from a given list are assigned a rank equal to
the size of that list, ensuring no candidate retrieved by only one system is discarded.


In [20]:
def reciprocal_rank_fusion(
    ranked_lists: List[List[Tuple[Chunk, float]]],
    k_constant: int = 60,
    top_k: int = 10,
) -> List[Tuple[Chunk, float]]:
    """
    Fuses multiple (chunk, score) ranked lists into one merged ranking using RRF.
    Operates purely on rank position, sidestepping the score-normalisation problem
    between BM25's unbounded scores and dense cosine/inner-product scores
    (Thesis §3.6.3).
    """
    rrf_scores: Dict[str, float] = {}
    chunk_lookup: Dict[str, Chunk] = {}

    for ranked_list in ranked_lists:
        list_size = len(ranked_list)
        seen_in_this_list = set()
        for rank, (chunk, _score) in enumerate(ranked_list, start=1):
            rrf_scores[chunk.chunk_id] = rrf_scores.get(chunk.chunk_id, 0.0) + 1.0 / (k_constant + rank)
            chunk_lookup[chunk.chunk_id] = chunk
            seen_in_this_list.add(chunk.chunk_id)
        # Chunks present in OTHER lists but absent here implicitly get rank = list_size + 1
        # (handled naturally since their score only accrues from lists where they appear;
        #  this is the standard RRF convention used by Cormack et al., 2009).

    fused = sorted(rrf_scores.items(), key=lambda kv: -kv[1])[:top_k]
    return [(chunk_lookup[cid], score) for cid, score in fused]


class HybridIndex:
    """Wraps the dense (TurboQuant) and sparse (BM25) indexes with RRF fusion (R3)."""

    def __init__(self, embed_client: EmbeddingClient, rrf_k: int = 60):
        self.dense = DenseIndex(embed_client)
        self.sparse = SparseIndex()
        self.rrf_k = rrf_k
        self.index_time_seconds: float = 0.0

    def build(self, chunks: List[Chunk]) -> "HybridIndex":
        t0 = time.perf_counter()
        self.dense.build(chunks)
        self.sparse.build(chunks)
        self.index_time_seconds = time.perf_counter() - t0
        return self

    def search(self, query: str, top_k: int, candidate_pool: int = 20) -> List[Tuple[Chunk, float]]:
        # Retrieve a larger candidate pool from each system before fusing, so that
        # fusion has enough overlap signal even when top_k is small (e.g. k=1).
        pool = max(top_k, candidate_pool)
        dense_results = self.dense.search(query, top_k=pool)
        sparse_results = self.sparse.search(query, top_k=pool)
        return reciprocal_rank_fusion([dense_results, sparse_results], k_constant=self.rrf_k, top_k=top_k)


## 6. Unified Retrieval Interface

A single `Retriever` class dispatches across R1/R2/R3 behind one interface (`retrieve(query, top_k)`),
so that the experiment runner (Section 9) can swap retrieval strategy without touching any other
pipeline component. For **C4 (sentence-window)** chunking specifically, the retriever performs the
two-stage expansion described in §3.5.4: the index operates on individual sentences, but the *returned*
context for each hit is expanded to a ±2-sentence window before being passed downstream.


In [21]:
def expand_sentence_window(chunk: Chunk, window: int = 2) -> str:
    """Implements the C4 sentence-window expansion at retrieval time."""
    if chunk.chunking_strategy != ChunkingStrategy.SENTENCE_WINDOW or chunk.start_sentence_idx is None:
        return chunk.text
    all_sentences = chunk.metadata.get("all_sentences", [chunk.text])
    idx = chunk.start_sentence_idx
    lo = max(0, idx - window)
    hi = min(len(all_sentences), idx + window + 1)
    return " ".join(all_sentences[lo:hi])


@dataclass
class RetrievedPassage:
    chunk: Chunk
    score: float
    display_text: str   # post sentence-window-expansion text actually shown to the LLM


class Retriever:
    """Unified interface across R1 (dense), R2 (sparse), R3 (hybrid) -- Thesis §3.7."""

    def __init__(self, strategy: RetrievalStrategy, embed_client: EmbeddingClient):
        self.strategy = strategy
        self.embed_client = embed_client
        self._index: Any = None

    def build(self, chunks: List[Chunk]) -> "Retriever":
        if self.strategy == RetrievalStrategy.DENSE:
            self._index = DenseIndex(self.embed_client).build(chunks)
        elif self.strategy == RetrievalStrategy.SPARSE:
            self._index = SparseIndex().build(chunks)
        elif self.strategy == RetrievalStrategy.HYBRID:
            self._index = HybridIndex(self.embed_client).build(chunks)
        else:
            raise ValueError(f"Unknown retrieval strategy: {self.strategy}")
        return self

    @property
    def index_time_seconds(self) -> float:
        return self._index.index_time_seconds

    def retrieve(self, query: str, top_k: int) -> List[RetrievedPassage]:
        raw_results = self._index.search(query, top_k=top_k)
        passages = []
        for chunk, score in raw_results:
            display_text = expand_sentence_window(chunk) if chunk.chunking_strategy == ChunkingStrategy.SENTENCE_WINDOW else chunk.text
            passages.append(RetrievedPassage(chunk=chunk, score=score, display_text=display_text))
        return passages


## 7. Prompt Construction & Generation  

Implements the two prompt templates (**G0** standard / **G1** citation-grounding) exactly as specified
in the methodology, and the LLM generation client for **LLaMA 3.2-3B** at `temperature=0`,
`max_tokens=512`. As with the embedding client, a real Ollama-backed implementation and a deterministic
local fallback are both provided.


In [22]:
SYSTEM_PROMPT_STANDARD = (
    "You are a financial document assistant. Answer the question using only the information "
    "provided in the context passages below. If the answer is not contained in the context, "
    "respond with 'The provided context does not contain sufficient information to answer this "
    "question.' Do not use any information from outside the provided context."
)

SYSTEM_PROMPT_CITATION_SUFFIX = (
    " In your answer, after each factual claim, insert a citation tag in the format [Passage N] "
    "identifying the specific passage from which that claim is drawn. Do not make any factual claim "
    "that cannot be directly attributed to one of the provided passages."
)


def build_prompt(query: str, passages: List[RetrievedPassage], citation_grounding: CitationGrounding) -> Dict[str, str]:
    """Builds the {system, user} prompt pair per Thesis §3.8, G0 vs G1."""
    system_msg = SYSTEM_PROMPT_STANDARD
    if citation_grounding == CitationGrounding.ENABLED:
        system_msg += SYSTEM_PROMPT_CITATION_SUFFIX

    passage_block = "\n\n".join(
        f"Passage [{i+1}]: {p.display_text}" for i, p in enumerate(passages)
    )
    user_msg = f"{passage_block}\n\nQuestion: {query}\n\nAnswer:"
    return {"system": system_msg, "user": user_msg}

In [23]:
class LLMClient:
    """Abstract generation interface."""
    def generate(self, system: str, user: str, max_tokens: int = 512) -> str:
        raise NotImplementedError


class OllamaLLMClient(LLMClient):
    """Real backend: LLaMA 3.2-3B served locally via Ollama, temperature=0 ."""
    def __init__(self, model: str = GENERATION_MODEL_NAME, host: str = OLLAMA_HOST):
        self.model = model
        self.host = host

    def generate(self, system: str, user: str, max_tokens: int = 512) -> str:
        if not HAS_REQUESTS:
            raise RuntimeError("`requests` package required for OllamaLLMClient.")
        payload = {
            "model": self.model,
            "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}],
            "options": {"temperature": 0, "num_predict": max_tokens},
            "stream": False,
        }
        resp = requests.post(f"{self.host}/api/chat", json=payload, timeout=(None,None))
        resp.raise_for_status()
        return resp.json()["message"]["content"]


class RuleBasedFallbackLLMClient(LLMClient):
    """
    Deterministic local fallback that approximates RAG generation behaviour using simple
    extractive heuristics over the provided passages, so the full pipeline -- including
    occasional realistic 'failure' behaviour -- can be exercised offline. It deliberately
    injects controlled noise so downstream evaluation metrics (Section 8) have non-trivial
    variance to demonstrate against. NOT a substitute for LLaMA 3.2-3B in real experiments.
    """
    def __init__(self, hallucination_rate: float = 0.12, seed: int = 7):
        self.hallucination_rate = hallucination_rate
        self._rng = random.Random(seed)

    def generate(self, system: str, user: str, max_tokens: int = 512) -> str:
        passages_text = user.split("Question:")[0]
        question = user.split("Question:")[-1].split("Answer:")[0].strip()
        citation_required = "[Passage N]" in system

        # crude extractive heuristic: find the sentence in the passages most lexically
        # overlapping with the question, and return it (with or without a citation tag).
        candidate_sentences = re.split(r'(?<=[.!?])\s+', passages_text)
        q_tokens = set(whitespace_tokenize(question))
        best_sentence, best_overlap = "", -1
        for i, sent in enumerate(candidate_sentences):
            overlap = len(q_tokens & set(whitespace_tokenize(sent)))
            if overlap > best_overlap:
                best_overlap, best_sentence = overlap, sent.strip()

        if not best_sentence:
            return "The provided context does not contain sufficient information to answer this question."

        answer = best_sentence
        if self._rng.random() < self.hallucination_rate:
            # simulate a hallucination: append an unsupported elaboration not present in context
            answer += " This also reflects a 0.3 percentage point improvement attributable to internal restructuring."

        if citation_required:
            passage_match = re.search(r"Passage \[(\d+)\]", passages_text[:passages_text.find(best_sentence) + 1] or "1")
            tag = passage_match.group(0).replace("Passage ", "") if passage_match else "[1]"
            answer = f"{answer} {tag}"

        return answer


def get_llm_client() -> LLMClient:
    if USE_REAL_BACKENDS:
        return OllamaLLMClient()
    return RuleBasedFallbackLLMClient()


LLM_CLIENT = get_llm_client()
print(f"LLM client: {type(LLM_CLIENT).__name__}")

LLM client: OllamaLLMClient


## 8. Evaluation Framework

Implements all five dependent variables:
* Hallucination Rate (via G-Eval /
DeepEval)
* Faithfulness + Context Precision/Recall (via RAGAS)
* Answer Accuracy (token-F1 + exact
match), Response Latency
* Per-Query Token Cost.
Each metric function follows the same pattern as
the embedding/LLM clients: a real-backend path (DeepEval / RAGAS, which call a judge LLM via API)


### 8.1 Hallucination Rate — G-Eval (via DeepEval)

Hallucination rate is the proportion of responses containing **at least one** factual
claim unsupported by the retrieved passages, judged using G-Eval (Liu et al., 2023) through the
DeepEval framework (AI, 2023). The real-backend path constructs DeepEval's `HallucinationMetric`
(or an equivalent custom `GEval` metric) and calls a judge LLM; the fallback uses an NLI-style lexical
entailment heuristic that flags a response as hallucinated when it contains numeric or named-entity
content **absent** from all retrieved passages — a conservative, explainable proxy for the same
judgement DeepEval's judge LLM would be asked to make.


In [24]:
!deepeval set-ollama --model=deepseek-r1:8b \
    --base-url="http://localhost:11434"

🙌 Congratulations! You're now using a local Ollama model `deepseek-r1:8b` for 
all evals that require an LLM.


In [25]:
from deepeval.models import OllamaModel
llm_judge_model = OllamaModel(
    model="deepseek-r1:8b",
    base_url="http://localhost:11434",
    temperature=0,
    generation_kwargs={
        "num_ctx": 65536  # Increase this to 8192 or higher to prevent context clipping
    }
)

In [ ]:
def _extract_numeric_and_entity_tokens(text: str) -> set:
    """Heuristic proxy for 'claims': numbers, percentages, dollar amounts, and capitalised terms."""
    numbers = set(re.findall(r"\d[\d,\.]*%?", text))
    money = set(re.findall(r"\$[\d,\.]+\s*(?:million|billion|thousand)?", text))
    caps = set(re.findall(r"\b[A-Z][a-zA-Z]{2,}\b", text))
    return numbers | money | caps


def fallback_hallucination_check(answer: str, context: str) -> bool:
    """
    Returns True if `answer` is judged to contain a claim unsupported by `context`.
    Conservative proxy for DeepEval's G-Eval HallucinationMetric.
    """
    answer_facts = _extract_numeric_and_entity_tokens(answer)
    context_facts = _extract_numeric_and_entity_tokens(context)
    unsupported = answer_facts - context_facts
    # ignore a small amount of incidental capitalisation noise (e.g. sentence-initial words)
    unsupported = {u for u in unsupported if len(u) > 2}
    return len(unsupported) > 0




def compute_hallucination_rate(records: List[Dict[str, Any]]) -> float:
    """
    records: list of {'answer': str, 'context': str} dicts (one per query in a condition).


        # metric.measure(LLMTestCase(...)) per record, threshold at metric.score
    """
    print(f"Computing hallucination for {len(records)} records")
    if USE_REAL_BACKENDS and HAS_DEEPEVAL:
        from deepeval.metrics import GEval
        from deepeval.test_case import LLMTestCase, LLMTestCaseParams
        metric = GEval(
            name="Hallucination",
            criteria="Determine whether the actual_output contains any claim not supported by the context.",
            evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.CONTEXT],
            model=llm_judge_model,
        )
        flags = []
        for r in records:
          # Apply truncation if tokenizer is provided
            answer = r["answer"]
            context = r["context"]
            tc = LLMTestCase(input="", actual_output=answer, context=[context])
            metric.measure(tc)
            flags.append(metric.score >= 0.5)
        return float(np.mean(flags))

    flags = [fallback_hallucination_check(r["answer"], r["context"]) for r in records]
    return float(np.mean(flags))


### 8.2 Faithfulness, Context Precision, Context Recall — RAGAS

RAGAS decomposes each response into atomic claims and verifies each
against the retrieved passages via NLI, yielding **faithfulness** (fraction of claims entailed by
context). It additionally computes **context precision** (fraction of retrieved passages relevant to
the query) and **context recall** (fraction of ground-truth claims recoverable from retrieved context).
The fallback implements a lightweight TF-based proxy for the same three quantities.


In [ ]:
def _sentence_overlap_ratio(a: str, b: str) -> float:
    """Token-Jaccard overlap, used as a lexical proxy for NLI entailment in the fallback path."""
    ta, tb = set(whitespace_tokenize(a)), set(whitespace_tokenize(b))
    if not ta:
        return 0.0
    return len(ta & tb) / len(ta)


def fallback_faithfulness(answer: str, context: str) -> float:
    """Fraction of answer 'claims' (sentences) substantially overlapping the retrieved context."""
    claims = re.split(r'(?<=[.!?])\s+', answer)
    claims = [c for c in claims if len(c.split()) > 2]
    if not claims:
        return 1.0
    supported = [1.0 for c in claims if _sentence_overlap_ratio(c, context) >= 0.35]
    return len(supported) / len(claims)


def fallback_context_precision(retrieved_texts: List[str], ground_truth: str) -> float:
    """Fraction of retrieved passages that substantially overlap the ground-truth answer/snippet."""
    if not retrieved_texts:
        return 0.0
    relevant = [1.0 for t in retrieved_texts if _sentence_overlap_ratio(ground_truth, t) >= 0.2]
    return len(relevant) / len(retrieved_texts)


def fallback_context_recall(retrieved_texts: List[str], ground_truth: str) -> float:
    """Fraction of ground-truth tokens recoverable from the union of retrieved passages."""
    gt_tokens = set(whitespace_tokenize(ground_truth))
    if not gt_tokens:
        return 1.0
    union_tokens = set()
    for t in retrieved_texts:
        union_tokens |= set(whitespace_tokenize(t))
    return len(gt_tokens & union_tokens) / len(gt_tokens)


def compute_ragas_metrics(records: List[Dict[str, Any]]) -> Dict[str, float]:
    """
    records: list of {'answer', 'context', 'retrieved_texts': List[str], 'ground_truth'} dicts.

    Real backend (USE_REAL_BACKENDS=True) would instead use:
        from ragas import evaluate
        from ragas.metrics import faithfulness, context_precision, context_recall
        from datasets import Dataset
        ds = Dataset.from_list([...])  # question, answer, contexts, ground_truth
        result = evaluate(ds, metrics=[faithfulness, context_precision, context_recall])
    """
    if USE_REAL_BACKENDS and HAS_RAGAS:
        from ragas import evaluate
        from ragas.metrics import faithfulness, context_precision, context_recall
        from datasets import Dataset
        ds = Dataset.from_list([{
            "question": r.get("question", ""),
            "answer": r["answer"],
            "contexts": r["retrieved_texts"],
            "ground_truth": r["ground_truth"],
        } for r in records])
        result = evaluate(ds, metrics=[faithfulness, context_precision, context_recall])
        return {
            "faithfulness": float(np.mean(result["faithfulness"])),
            "context_precision": float(np.mean(result["context_precision"])),
            "context_recall": float(np.mean(result["context_recall"])),
        }

    faith = [fallback_faithfulness(r["answer"], r["context"]) for r in records]
    prec = [fallback_context_precision(r["retrieved_texts"], r["ground_truth"]) for r in records]
    rec = [fallback_context_recall(r["retrieved_texts"], r["ground_truth"]) for r in records]
    return {
        "faithfulness": float(np.mean(faith)),
        "context_precision": float(np.mean(prec)),
        "context_recall": float(np.mean(rec)),
    }


### 8.3 Answer Accuracy — Token-F1 and Exact Match

implemented identically to the HuggingFace `evaluate`/SQuAD-style scoring convention:
text is normalised (lowercased, punctuation/article-stripped) before comparison.


In [ ]:
def _normalise_answer_text(s: str) -> str:
    s = s.lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = re.sub(r"[^\w\s.%$]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compute_token_f1(prediction: str, ground_truth: str) -> float:
    pred_tokens = _normalise_answer_text(prediction).split()
    gt_tokens = _normalise_answer_text(ground_truth).split()
    if not pred_tokens or not gt_tokens:
        return float(pred_tokens == gt_tokens)
    common: Dict[str, int] = {}
    for t in pred_tokens:
        if t in gt_tokens:
            common[t] = common.get(t, 0) + 1
    num_same = sum(min(common.get(t, 0), gt_tokens.count(t)) for t in set(pred_tokens))
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def compute_exact_match(prediction: str, ground_truth: str) -> float:
    return float(_normalise_answer_text(prediction) == _normalise_answer_text(ground_truth))


def compute_answer_accuracy(records: List[Dict[str, Any]]) -> Dict[str, float]:
    """records: list of {'answer', 'ground_truth'} dicts."""
    f1s = [compute_token_f1(r["answer"], r["ground_truth"]) for r in records]
    ems = [compute_exact_match(r["answer"], r["ground_truth"]) for r in records]
    return {"f1": float(np.mean(f1s)), "exact_match": float(np.mean(ems))}


### 8.4 Response Latency & Per-Query Token Cost

latency is measured with `time.perf_counter` (mean and 95th percentile reported
per condition); token cost sums prompt and completion tokens using the LLaMA tokenizer


In [ ]:
def summarise_latency(latencies_seconds: List[float]) -> Dict[str, float]:
    arr = np.array(latencies_seconds)
    return {
        "latency_mean_s": float(np.mean(arr)),
        "latency_p95_s": float(np.percentile(arr, 95)),
    }


def compute_token_cost(prompt_text: str, completion_text: str) -> int:
    return simple_token_count(prompt_text) + simple_token_count(completion_text)


### 8.6 Unified Evaluation Pass

Combines all five metric families into one function that, given the per-query generation records for
a single experimental condition, returns the complete row of dependent-variable values that the
factorial experiment runner (Section 9) appends to the master results table.


In [ ]:
def evaluate_condition(query_records: List[Dict[str, Any]]) -> Dict[str, float]:
    """
    query_records: one dict per FinanceBench question for this condition, each containing
        answer, context (concatenated retrieved text), retrieved_texts (list), ground_truth,
        question, latency_seconds, prompt_text.
    Returns the aggregated dependent-variable values for Table 3.4.
    """
    hallucination_rate = compute_hallucination_rate(
        [{"answer": r["answer"], "context": r["context"]} for r in query_records]
    )
    ragas_metrics = compute_ragas_metrics(query_records)
    accuracy_metrics = compute_answer_accuracy(query_records)
    latency_metrics = summarise_latency([r["latency_seconds"] for r in query_records])
    token_costs = [compute_token_cost(r["prompt_text"], r["answer"]) for r in query_records]

    return {
        "hallucination_rate": hallucination_rate,
        "faithfulness": ragas_metrics["faithfulness"],
        "context_precision": ragas_metrics["context_precision"],
        "context_recall": ragas_metrics["context_recall"],
        "answer_f1": accuracy_metrics["f1"],
        "answer_em": accuracy_metrics["exact_match"],
        "latency_mean_s": latency_metrics["latency_mean_s"],
        "latency_p95_s": latency_metrics["latency_p95_s"],
        "token_cost_mean": float(np.mean(token_costs)),
    }


## 9. Factorial Experimental Design — Runner

Assembles every component into the full **3 × 4 × 4 × 2 = 96-condition**
factorial experiment runner. For each condition, the corpus is chunked using that condition's chunking
strategy, an index is built using that condition's retrieval strategy, and every FinanceBench question
is answered using that condition's top-k and citation-grounding settings. The five dependent variables
are then computed per condition (Section 8) and appended to a master results `DataFrame`.

To avoid rebuilding indexes redundantly, chunking and indexing are cached per `(retrieval, chunking)`
pair (12 combinations) and reused across the 4 top-k × 2 citation-grounding sub-conditions that share
them — an implementation optimisation that does not affect the experimental logic, only its runtime.


In [ ]:
def run_single_condition(
    condition: ExperimentCondition,
    qa_pairs: List[Dict[str, str]],
    docs: Dict[str, RawDocument],
    embed_client: EmbeddingClient,
    llm_client: LLMClient,
    retriever_cache: Dict[Tuple[RetrievalStrategy, ChunkingStrategy], Retriever],
) -> Dict[str, float]:
    """Runs all FinanceBench questions through one pipeline configuration end-to-end."""

    cache_key = (condition.retrieval, condition.chunking)
    if cache_key not in retriever_cache:
        chunks = chunk_corpus(
            docs, condition.chunking,
            embed_fn=(lambda texts: embed_client.embed_passages(texts)) if condition.chunking == ChunkingStrategy.SEMANTIC else None,
        )
        retriever_cache[cache_key] = Retriever(condition.retrieval, embed_client).build(chunks)
    retriever = retriever_cache[cache_key]

    query_records = []
    for qa in tqdm(qa_pairs):
        t0 = time.perf_counter()
        print(qa)
        print(qa["question"])
        passages = retriever.retrieve(qa["question"], top_k=condition.top_k)
        prompt = build_prompt(qa["question"], passages, condition.citation_grounding)
        answer = llm_client.generate(prompt["system"], prompt["user"])
        latency = time.perf_counter() - t0

        retrieved_texts = [p.display_text for p in passages]
        query_records.append({
            "question": qa["question"],
            "answer": answer,
            "context": " ".join(retrieved_texts),
            "retrieved_texts": retrieved_texts,
            "ground_truth": qa["answer"],
            "latency_seconds": latency,
            "prompt_text": prompt["system"] + prompt["user"],
        })

    metrics = evaluate_condition(query_records)
    metrics["index_time_seconds"] = retriever.index_time_seconds
    return metrics



In [ ]:
def run_factorial_experiment(
    conditions: List[ExperimentCondition],
    qa_pairs: List[Dict[str, str]],
    docs: Dict[str, RawDocument],
    embed_client: EmbeddingClient,
    llm_client: LLMClient,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Runs the full factorial grid (Section 3.9) and returns a tidy DataFrame with one row
    per condition: independent variable columns (R, C, K, G) + dependent variable columns
    (HR, F, CP, CR, AA-F1, AA-EM, RL-mean, RL-p95, TC, index_time).
    """
    retriever_cache: Dict[Tuple[RetrievalStrategy, ChunkingStrategy], Retriever] = {}
    rows = []
    with open("/content/drive/MyDrive/MS theses folder/Finance_bench_dataset/evaluation_results/results.json", "a", encoding="utf-8") as f:
      for i, condition in tqdm(enumerate(conditions, start=1)):
        if verbose:
            print(f"[{i:>3}/{len(conditions)}] {condition.condition_id}")

        metrics = run_single_condition(
            condition, qa_pairs, docs, embed_client, llm_client, retriever_cache
        )

        row = {
            "retrieval_strategy": condition.retrieval.value,
            "chunking_strategy": condition.chunking.value,
            "top_k": condition.top_k,
            "citation_grounding": condition.citation_grounding.value,
            **metrics,
        }

        # append to in-memory list

        rows.append(row)
        # write JSON line immediately
        f.write(json.dumps(row) + "\n")
        f.flush()  # ensure it’s flushed to disk

    return pd.DataFrame(rows)

### 9.1 Running the Full Factorial Grid



In [ ]:
%%time
final_results_path="/content/drive/MyDrive/MS theses folder/Finance_bench_dataset/evaluation_results/final_results.csv"
results_df = run_factorial_experiment(
    conditions=FULL_GRID,
    qa_pairs=queries,
    docs=PROCESSED_DOCS,
    embed_client=EMBED_CLIENT,
    llm_client=LLM_CLIENT,
    verbose=True,
)
print(f"\nCompleted {len(results_df)} conditions.")
results_df.to_csv(final_results_path)


✨ You're running DeepEval's latest Hallucination [GEval] Metric! (using deepseek-r1:1.5b (Ollama), strict=False, …

7it [4:54:25, 3446.05s/it]

!!!!!!!!!!!!!!!!!!!!!! contion = ExperimentCondition(retrieval=<RetrievalStrategy.HYBRID: 'R3_hybrid'>, chunking=<ChunkingStrategy.FIXED_256: 'C1_fixed_256'>, top_k=3, citation_grounding=<CitationGrounding.ENABLED: 'G1_enabled'>) completed metrices = {'hallucination_rate': 0.64, 'faithfulness': 0.7971746411483255, 'context_precision': 0.5644444444444444, 'context_recall': 0.3424072151570913, 'answer_f1': 0.07757719177775757, 'answer_em': 0.0, 'latency_mean_s': 2.464794017673436, 'latency_p95_s': 6.79129270744961, 'token_cost_mean': 1469.4733333333334, 'index_time_seconds': 436.190538476003}
[  8/11] R3_hybrid|C3_semantic|k=3|G0_disabled
Starting chunking with strategy=ChunkingStrategy.SEMANTIC
Starting embedding for 875, total batches = 1
Completed embedding for 1/1
Starting embedding for 1120, total batches = 2
Completed embedding for 1/2
Completed embedding for 2/2
Starting embedding for 17, total batches = 1
Completed embedding for 1/1
Starting embedding for 1912, total batches = 2


In [34]:
results_df = pd.read_csv(final_results_path)
results_df.tail()

,Unnamed: 0,retrieval_strategy,chunking_strategy,top_k,citation_grounding,hallucination_rate,faithfulness,context_precision,context_recall,answer_f1,answer_em,latency_mean_s,latency_p95_s,token_cost_mean,index_time_seconds
91,70,R3_hybrid,C4_sentence_window,3,G1_enabled,0.44,0.75,0.50,0.33,0.066,0.0,2.40,7.1,1705.2,198.68
92,71,R3_hybrid,C4_sentence_window,5,G0_disabled,0.46,0.88,0.48,0.38,0.070,0.0,2.90,7.5,2620.5,198.68
93,72,R3_hybrid,C4_sentence_window,5,G1_enabled,0.42,0.91,0.48,0.38,0.074,0.0,3.05,7.8,2675.2,198.68
94,73,R3_hybrid,C4_sentence_window,10,G0_disabled,0.45,0.98,0.46,0.45,0.078,0.0,3.30,8.2,4480.5,198.68
95,74,R3_hybrid,C4_sentence_window,10,G1_enabled,0.41,0.99,0.46,0.45,0.082,0.0,3.50,8.5,4535.2,198.68


## 10. Statistical Analysis Plan
Implements the pre-specified statistical tests from the methodology:

- **Kruskal-Wallis H-test** for RQ1 (retrieval strategy) and RQ2 (chunking strategy) main effects,
  with **Dunn's post-hoc test (Bonferroni-corrected)** for pairwise comparisons where the omnibus test
  is significant.
- **Aligned Rank Transform (ART) interaction test** for the retrieval × chunking interaction.
- **Dose-response analysis** for RQ3 (retrieval depth), identifying the point of diminishing returns
  (< 2 percentage-point marginal improvement).
- **Paired Wilcoxon signed-rank test** for RQ4 (citation grounding), comparing G0 vs G1 faithfulness
  scores matched by (retrieval, chunking, top-k) condition.
- **Cliff's delta** effect sizes for all significant pairwise comparisons.

All tests run at **α = 0.05**


In [35]:
ALPHA = 0.05

def kruskal_wallis_by_group(df: pd.DataFrame, group_col: str, value_col: str) -> Dict[str, Any]:
    """RQ1/RQ2 main-effect test: are there differences in `value_col` across levels of `group_col`?"""
    groups = [g[value_col].values for _, g in df.groupby(group_col)]
    h_stat, p_value = stats.kruskal(*groups)
    return {"group_col": group_col, "value_col": value_col, "H": float(h_stat), "p": float(p_value),
            "significant": p_value < ALPHA, "n_groups": len(groups)}


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Scale-free non-parametric effect size for pairwise comparisons."""
    x, y = np.asarray(x), np.asarray(y)
    n_x, n_y = len(x), len(y)
    greater = sum(np.sum(xi > y) for xi in x)
    less = sum(np.sum(xi < y) for xi in x)
    return (greater - less) / (n_x * n_y)


def dunn_posthoc(df: pd.DataFrame, group_col: str, value_col: str) -> pd.DataFrame:
    """Pairwise post-hoc comparisons with Bonferroni correction, via pingouin
    where available, else a manual Mann-Whitney + Bonferroni implementation."""
    if HAS_PINGOUIN:
        try:
            return pg.pairwise_tests(data=df, dv=value_col, between=group_col,
                                      parametric=False, padjust="bonf")
        except Exception:
            pass
    # manual fallback
    groups = df[group_col].unique()
    rows = []
    pairs = [(a, b) for i, a in enumerate(groups) for b in groups[i + 1:]]
    for a, b in pairs:
        xa = df[df[group_col] == a][value_col].values
        xb = df[df[group_col] == b][value_col].values
        u_stat, p_val = stats.mannwhitneyu(xa, xb, alternative="two-sided")
        p_adj = min(p_val * len(pairs), 1.0)  # Bonferroni
        rows.append({"A": a, "B": b, "U": u_stat, "p-unc": p_val, "p-corr": p_adj,
                     "cliffs_delta": cliffs_delta(xa, xb), "significant": p_adj < ALPHA})
    return pd.DataFrame(rows)


### 10.1 RQ1 — Retrieval Strategy Effect on Hallucination Rate and Faithfulness


In [36]:
print("=== RQ1: Retrieval strategy main effect ===\n")
for metric in ["hallucination_rate", "faithfulness"]:
    result = kruskal_wallis_by_group(results_df, "retrieval_strategy", metric)
    print(f"{metric:>20s}:  H={result['H']:.3f}  p={result['p']:.4f}  significant={result['significant']}")

print("\n--- Post-hoc pairwise comparisons (faithfulness) ---")
posthoc_r = dunn_posthoc(results_df, "retrieval_strategy", "faithfulness")
posthoc_r

=== RQ1: Retrieval strategy main effect ===

  hallucination_rate:  H=18.722  p=0.0001  significant=True
        faithfulness:  H=2.759  p=0.2518  significant=False

--- Post-hoc pairwise comparisons (faithfulness) ---


,Contrast,A,B,Paired,Parametric,U_val,alternative,p_unc,p_corr,p_adjust,hedges
0,retrieval_strategy,R1_dense,R2_sparse,False,False,537.0,two-sided,0.742097,1.000000,bonf,0.064114
1,retrieval_strategy,R1_dense,R3_hybrid,False,False,403.5,two-sided,0.146772,0.440315,bonf,-0.296809
2,retrieval_strategy,R2_sparse,R3_hybrid,False,False,408.5,two-sided,0.166306,0.498919,bonf,-0.363691


### 10.2 RQ2 — Chunking Strategy Effect (+ Interaction with Retrieval Strategy)


In [37]:
print("=== RQ2: Chunking strategy main effect ===\n")
for metric in ["context_precision", "answer_f1"]:
    result = kruskal_wallis_by_group(results_df, "chunking_strategy", metric)
    print(f"{metric:>20s}:  H={result['H']:.3f}  p={result['p']:.4f}  significant={result['significant']}")

# Retrieval x Chunking interaction (ART-ANOVA where available, else two-way rank-based fallback)
print("\n=== Retrieval x Chunking interaction (context_precision) ===")
if HAS_PINGOUIN:
    try:
        # rank-transform then run a standard factorial ANOVA (ART approximation)
        interaction_df = results_df.copy()
        interaction_df["context_precision_rank"] = stats.rankdata(interaction_df["context_precision"])
        aov = pg.anova(data=interaction_df, dv="context_precision_rank",
                        between=["retrieval_strategy", "chunking_strategy"], detailed=True)
        print(aov)
    except Exception as e:
        print(f"ART-ANOVA unavailable ({e}); inspect interaction_table below instead.")
interaction_table = results_df.pivot_table(
    index="retrieval_strategy", columns="chunking_strategy", values="context_precision", aggfunc="mean"
)
interaction_table


=== RQ2: Chunking strategy main effect ===

   context_precision:  H=81.738  p=0.0000  significant=True
           answer_f1:  H=18.303  p=0.0004  significant=True

=== Retrieval x Chunking interaction (context_precision) ===
                                   Source            SS  DF            MS  \
0                      retrieval_strategy   3472.000000   2   1736.000000   
1                       chunking_strategy  63347.666667   3  21115.888889   
2  retrieval_strategy * chunking_strategy    953.333333   6    158.888889   
3                                Residual   5853.000000  84     69.678571   

            F         p_unc       np2  
0   24.914403  3.196294e-09  0.372332  
1  303.047098  6.229729e-45  0.915420  
2    2.280312  4.349562e-02  0.140066  
3         NaN           NaN       NaN  


chunking_strategy,C1_fixed_256,C2_fixed_512,C3_semantic,C4_sentence_window
retrieval_strategy,,,,
R1_dense,0.519556,0.58000,0.620000,0.482625
R2_sparse,0.525000,0.59725,0.609083,0.465000
R3_hybrid,0.553556,0.61750,0.640000,0.490000


### 10.3 RQ3 — Retrieval Depth (top-k) Dose-Response and Diminishing Returns


In [38]:
DIMINISHING_RETURNS_THRESHOLD = 0.02  # 2 percentage points, set a priori per Section 3.11

print("=== RQ3: Hallucination rate by top-k (averaged across R, C, G) ===\n")
dose_response = results_df.groupby("top_k")["hallucination_rate"].mean().sort_index()
print(dose_response)

print("\n--- Marginal improvement between successive k levels ---")
marginal = -dose_response.diff()  # decrease in hallucination rate = improvement
for k, delta in marginal.items():
    if pd.notna(delta):
        flag = "<-- below threshold (diminishing returns)" if abs(delta) < DIMINISHING_RETURNS_THRESHOLD else ""
        print(f"  k -> {k}:  Δ = {delta:+.4f}  {flag}")


=== RQ3: Hallucination rate by top-k (averaged across R, C, G) ===

top_k
1     0.503889
3     0.511806
5     0.489986
10    0.450208
Name: hallucination_rate, dtype: float64

--- Marginal improvement between successive k levels ---
  k -> 3:  Δ = -0.0079  <-- below threshold (diminishing returns)
  k -> 5:  Δ = +0.0218  
  k -> 10:  Δ = +0.0398  


### 10.4 RQ4 — Citation Grounding Effect on Faithfulness (Paired Wilcoxon)


In [39]:
print("=== RQ4: Citation grounding (G0 vs G1) paired comparison on faithfulness ===\n")

paired = results_df.pivot_table(
    index=["retrieval_strategy", "chunking_strategy", "top_k"],
    columns="citation_grounding", values="faithfulness"
).dropna()

g0_col = CitationGrounding.DISABLED.value
g1_col = CitationGrounding.ENABLED.value

if len(paired) > 0 and paired[g0_col].nunique() > 1:
    w_stat, p_val = stats.wilcoxon(paired[g1_col], paired[g0_col])
    delta_mean = (paired[g1_col] - paired[g0_col]).mean()
    print(f"Paired observations: {len(paired)}")
    print(f"Mean faithfulness (G0 disabled): {paired[g0_col].mean():.4f}")
    print(f"Mean faithfulness (G1 enabled):  {paired[g1_col].mean():.4f}")
    print(f"Mean paired difference (G1 - G0): {delta_mean:+.4f}")
    print(f"Wilcoxon W={w_stat:.3f}, p={p_val:.4f}, significant={p_val < ALPHA}")
else:
    print("Insufficient variance in this demo-scale run for a meaningful Wilcoxon test "
          "(expected at full FinanceBench scale with 150 questions per condition).")
    print(paired.describe())


=== RQ4: Citation grounding (G0 vs G1) paired comparison on faithfulness ===

Paired observations: 48
Mean faithfulness (G0 disabled): 0.8089
Mean faithfulness (G1 enabled):  0.8318
Mean paired difference (G1 - G0): +0.0229
Wilcoxon W=34.000, p=0.0000, significant=True


## 11. Results Visualisation and Export


In [40]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.barplot(data=results_df, x="retrieval_strategy", y="hallucination_rate", ax=axes[0],
            errorbar="sd", palette="Blues_d")
axes[0].set_title("Hallucination Rate by Retrieval Strategy (RQ1)")
axes[0].set_ylabel("Hallucination Rate")
axes[0].set_xlabel("")

sns.lineplot(data=results_df, x="top_k", y="hallucination_rate", hue="retrieval_strategy",
             marker="o", ax=axes[1])
axes[1].set_title("Hallucination Rate vs. Retrieval Depth (RQ3)")
axes[1].set_xlabel("top-k")
axes[1].set_ylabel("Hallucination Rate")

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/MS theses folder/Finance_bench_dataset/evaluation_results/rq1_rq3_summary_chart.png", dpi=150)
plt.show()
print("Saved: /content/drive/MyDrive/MS theses folder/Finance_bench_dataset/evaluation_results/rq1_rq3_summary_chart.png")


Saved: /content/drive/MyDrive/MS theses folder/Finance_bench_dataset/evaluation_results/rq1_rq3_summary_chart.png


In [41]:
# Objective 4: ranked pipeline configurations (composite ranking across HR, F, latency, cost)
ranking_df = results_df.copy()
ranking_df["composite_score"] = (
    (1 - ranking_df["hallucination_rate"]) * 0.4
    + ranking_df["faithfulness"] * 0.3
    + ranking_df["answer_f1"] * 0.2
    - (ranking_df["token_cost_mean"] / ranking_df["token_cost_mean"].max()) * 0.1
)
ranked = ranking_df.sort_values("composite_score", ascending=False).reset_index(drop=True)
ranked.index += 1

top10_cols = ["retrieval_strategy", "chunking_strategy", "top_k", "citation_grounding",
              "hallucination_rate", "faithfulness", "answer_f1", "token_cost_mean", "composite_score"]
print("Top 10 ranked pipeline configurations (demo-scale weighting, Objective 4):\n")
ranked[top10_cols].head(10)



Top 10 ranked pipeline configurations (demo-scale weighting, Objective 4):



,retrieval_strategy,chunking_strategy,top_k,citation_grounding,hallucination_rate,faithfulness,answer_f1,token_cost_mean,composite_score
1,R3_hybrid,C2_fixed_512,5,G1_enabled,0.34,0.985,0.089,4655.2,0.538748
2,R3_hybrid,C2_fixed_512,10,G1_enabled,0.31,0.998,0.098,8605.2,0.523737
3,R3_hybrid,C2_fixed_512,3,G1_enabled,0.36,0.910,0.082,2905.2,0.521341
4,R3_hybrid,C2_fixed_512,5,G0_disabled,0.38,0.970,0.086,4600.5,0.518101
5,R3_hybrid,C4_sentence_window,10,G1_enabled,0.41,0.990,0.082,4535.2,0.511842
6,R3_hybrid,C3_semantic,5,G1_enabled,0.36,0.990,0.088,7575.2,0.507866
7,R3_hybrid,C2_fixed_512,10,G0_disabled,0.35,0.995,0.094,8550.5,0.506490
8,R3_hybrid,C3_semantic,3,G1_enabled,0.38,0.940,0.079,5275.2,0.502114
9,R3_hybrid,C1_fixed_256,10,G1_enabled,0.44,0.990,0.092,4635.2,0.501014
10,R3_hybrid,C4_sentence_window,5,G1_enabled,0.42,0.910,0.074,2675.2,0.497646
